# 🚀 NVIDIA A100 GPU Master Training Pipeline: 3 Machine Translation Architectures (v2 — Bug-Fixed)
### End-to-End Multilingual NMT (English ↔ Kiswahili ↔ Ekegusii) for Kenyan Public Service Announcements

This is a corrected rebuild of `A100_3_Architectures_Master_Pipeline_after_run.ipynb`. That run produced
near-zero scores across all three architectures (SacreBLEU 0.29–1.42, chrF 12.06–14.67) — numbers too low to
reflect a genuinely trained-but-mediocre model; they point to pipeline bugs. This notebook has **not been
executed** (no GPU / multi-hour budget in the environment that wrote it) — run it on your A100 instance and
share the new results back so the fix can be verified.

---
### 🔧 What changed from the previous run, and why

1. **Missing `forced_bos_token_id` at generation time (prime suspect for the near-zero scores).**
   The old notebook set `tokenizer.tgt_lang` only while *encoding training labels*. It never told
   `model.generate()` which language token to force as the first decoded token, so every `trainer.evaluate()`
   / `trainer.predict()` call fell back to the checkpoint's default decoder start token — meaning the model
   could have been trained correctly and still scored near-zero at eval time, in *any* of the 3 architectures.
   Fixed by setting `forced_bos_token_id` explicitly after building each model (see changelog item 9 below for
   *where* it has to be set — that moved again between v2 and v3 of this rebuild).

2. **Ekegusii silently reused Kiswahili's language tag (`swh_Latn`).**
   NLLB-200 has no native Ekegusii code, so the old notebook mapped both `Kiswahili` and `Ekegusii` to
   `swh_Latn`. That's not automatically fatal (every stage only ever decoded into "Ekegusii" here, so
   at least it was applied consistently) — but it means the model has no dedicated language head to learn
   Ekegusii-specific decoding behavior. Fixed by registering a **new** `guz_Latn` tag, resizing the model's
   embeddings, and warm-starting the new tag's embedding from `swh_Latn` (a linguistically motivated choice —
   Ekegusii and Kiswahili are both Bantu languages — rather than an accidental collision).

3. **No validation during the intermediate curriculum stages.**
   Architectures 2 and 3 only evaluated at the very end, so there was no way to see *which* stage helped or
   hurt. This rebuild evaluates chrF/BLEU against the shared PSA validation set after every stage, and plots
   a per-stage learning curve.

4. **Architecture 3's "unilingual pre-adaptation" stage taught the model to copy input to output verbatim**
   (`uni_sample['Ekegusii'] = uni_sample['English']`), tagged as if it were a real translation pair. Replaced
   with a proper denoising/domain-adaptive pretraining objective: corrupt English PSA sentences (word dropout
   + shuffling) and train the model to reconstruct the clean sentence, entirely in English
   (`eng_Latn -> eng_Latn`) — a standard self-supervised technique for domain adaptation, not a fake
   translation task into an unseen language.

5. Removed a broken `pip install chrf` line (not a real PyPI package — `evaluate.load('chrf')` fetches its
   own metric script and never needed it).

6. Added `generation_max_length`, `generation_num_beams`, `load_best_model_at_end`, and early stopping so
   training doesn't just run a fixed number of epochs regardless of whether validation is improving.

7. Broadened the dataset auto-resolver to also check this repo's actual folder names
   (`data_train_tringual/`, `data_train_bilingual/`, `data_train_unilingual/` at the repo root), while keeping
   the original recursive-glob fallback so it still works unmodified on Colab/Kaggle.

8. **(v2 patch)** The first version of this rebuild still crashed on a newer `transformers` release with
   `AttributeError: NllbTokenizer has no attribute additional_special_tokens` — the internal attributes
   (`lang_code_to_id`, `additional_special_tokens`, the `.src_lang`/`.tgt_lang` setters) that NLLB fine-tuning
   tutorials usually rely on are not stable public API and have shifted across `transformers` versions.
   Rewired tokenization to avoid all of them: `register_ekegusii_tag()` now uses the low-level, stable
   `tokenizer.add_tokens(...)` instead of `add_special_tokens(...)`, and `preprocess_nmt_function()` builds
   every sequence manually in NLLB's own convention — `[lang_tag_id, tok_1, ..., tok_n, eos_id]` — via plain
   `convert_tokens_to_ids()` lookups, instead of depending on `tokenizer.src_lang`/`tgt_lang` to inject the tag.
   Functionally identical to before, just independent of internals that may keep moving under you.

9. **(v3 patch)** Fixing #8 surfaced a second version-drift error: setting `model.config.forced_bos_token_id`
   now raises `ValueError: You have modified the pretrained model configuration to control generation ...
   This strategy to control generation is not supported anymore` on newer `transformers`. Generation
   parameters must now live on `model.generation_config`, a separate object from `model.config` — the two
   used to be interchangeable for this in older tutorials and no longer are. Fixed by setting
   `model.generation_config.forced_bos_token_id = EKEGUSII_TAG_ID` instead of `model.config....`.

10. **(v4 patch) `CUDA out of memory` on an 80GB A100 with only ~21MB free.** This one is *not* a code bug --
    the traceback showed **two other processes already holding ~32.65 GiB and ~31.25 GiB** on the same GPU
    before this notebook's model even finished loading. That's either another tenant/kernel sharing the card,
    or (very plausibly, given this notebook was re-run several times across earlier failed attempts) a stale
    kernel from a previous attempt that was never actually restarted. **Run `!nvidia-smi` first** and, if any
    listed process is one of your own old kernels, restart that kernel (not just re-run cells) before trying
    again. Regardless of the cause, this rebuild also adds real headroom: `gradient_checkpointing_enable()` +
    `enable_input_require_grads()` on every model, default batch size dropped 32→16, eval decoding dropped
    beam-search (4) → greedy (1), and each architecture's model/trainer is now explicitly `del`ed and
    `torch.cuda.empty_cache()`-ed before the next one loads (the 3 architectures were previously left resident
    on the GPU simultaneously across cells in the same kernel session).

*All 3 architectures are still evaluated on the same held-out 10% PSA test set for a fair comparison.*


## 1. Environment Setup & NVIDIA A100 GPU Verification
Installs dependencies and verifies GPU hardware. (Removed the invalid `chrf` pip package from the original —
see changelog above.)

In [ ]:
%pip install -U \
transformers \
peft \
datasets \
evaluate \
sacrebleu \
accelerate \
sentencepiece \
bitsandbytes \
matplotlib \
pandas \
numpy \
scikit-learn \
seaborn

import torch
import transformers
import peft
import datasets
import evaluate
import os
import sys
import glob
import random

print('=== GPU Hardware & Environment Info ===')
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    print('Total VRAM:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: Running on CPU. NVIDIA A100 GPU recommended for fast training.')


## 2. Dynamic Auto-Path Data Resolver & Stratified Architecture Splitting
Searches this repo's actual data folders first, then falls back to the original Colab/Kaggle-style paths and
a recursive glob, so the notebook runs unmodified in either environment. Performs Unicode cleaning and
constructs clean 80/10/10 train/val/test splits.

In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

def find_data_file(filename):
    possible_paths = [
        # This repo's actual layout (root-level folders)
        os.path.join('..', 'data_train_tringual', filename),
        os.path.join('..', 'data_train_bilingual', filename),
        os.path.join('..', 'data_train_unilingual', filename),
        os.path.join('data_train_tringual', filename),
        os.path.join('data_train_bilingual', filename),
        os.path.join('data_train_unilingual', filename),
        # Original Colab/Kaggle-style upload paths (kept for portability)
        os.path.join('..', 'data', 'data_trian_tringual', filename),
        os.path.join('..', 'data', 'data_train_bilingual', filename),
        os.path.join('..', 'data', 'data_train_unilingual', filename),
        os.path.join('..', 'data_trian_tringual', filename),
        os.path.join('data', 'data_trian_tringual', filename),
        os.path.join('data', 'data_train_bilingual', filename),
        os.path.join('data', 'data_train_unilingual', filename),
        filename,
        os.path.join('..', filename),
    ]
    for p in possible_paths:
        if os.path.exists(p):
            return p
    matches = glob.glob(f'**/{filename}', recursive=True) + glob.glob(f'../**/{filename}', recursive=True)
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Could not locate dataset file "{filename}" in current ({os.getcwd()}) or parent directory.')

def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'^["\'`]+|["\'`]+$', '', text).strip()
    return text

print('=== Auto-Resolving Dataset Paths & Creating Architecture Splits ===')
print('Jupyter Working Directory:', os.getcwd())

psa_path = find_data_file('psa.csv')
print(f'[FOUND] Trilingual PSA Dataset located at: "{psa_path}"')

splits_root = 'data_splits'
arch1_dir = os.path.join(splits_root, 'arch1_direct_trilingual_psa')
arch2_dir = os.path.join(splits_root, 'arch2_bilingual_bible_psa')
arch3_dir = os.path.join(splits_root, 'arch3_curriculum_learning')

for d in [arch1_dir, arch2_dir, arch3_dir]:
    os.makedirs(d, exist_ok=True)

psa_df = pd.read_csv(psa_path)
for col in ['English', 'Kiswahili', 'Ekegusii']:
    if col in psa_df.columns:
        psa_df[col] = psa_df[col].apply(clean_text)

psa_df = psa_df[(psa_df['English'].str.len() > 3) & (psa_df['Kiswahili'].str.len() > 3) & (psa_df['Ekegusii'].str.len() > 3)].drop_duplicates().reset_index(drop=True)

train_psa, temp_psa = train_test_split(psa_df, test_size=0.20, random_state=42)
val_psa, test_psa = train_test_split(temp_psa, test_size=0.50, random_state=42)

train_psa.to_csv(os.path.join(arch1_dir, 'train.csv'), index=False)
val_psa.to_csv(os.path.join(arch1_dir, 'val.csv'), index=False)
test_psa.to_csv(os.path.join(arch1_dir, 'test.csv'), index=False)

for d in [arch2_dir, arch3_dir]:
    val_psa.to_csv(os.path.join(d, 'val.csv'), index=False)
    test_psa.to_csv(os.path.join(d, 'test.csv'), index=False)

print(f'[OK] Splits created! Train: {len(train_psa)} | Val: {len(val_psa)} | Held-out test: {len(test_psa)}')
print('[CONFIRMED] Other datasets located:')
print(' -> Bilingual Web News:', find_data_file('English_Ekegusii_Web_News.csv'))
print(' -> Trilingual Bible:', find_data_file('bibile.csv'))
print(' -> Unilingual English:', find_data_file('english_unilingual.csv'))


## 3. NMT Tokenization, Model & Metric Utilities (fixed)
Registers a **dedicated** `guz_Latn` tag for Ekegusii instead of colliding with Kiswahili's `swh_Latn`, wires
`forced_bos_token_id` into both model config and generation calls, and centralizes model construction in
`build_model()` so every architecture gets the fix identically.

In [ ]:
import gc
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'facebook/nllb-200-distilled-600M'

# --- FIX: dedicated Ekegusii tag instead of reusing swh_Latn -----------------
EKEGUSII_TAG = 'guz_Latn'
LANG_TAGS = {'English': 'eng_Latn', 'Kiswahili': 'swh_Latn', 'Ekegusii': EKEGUSII_TAG}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# NOTE ON ROBUSTNESS: newer transformers releases have reshuffled a lot of the
# NllbTokenizer internals (`lang_code_to_id`, `additional_special_tokens`,
# `.src_lang` / `.tgt_lang` setters) -- the exact attributes available differ
# across versions and are not part of any stable public API. Rather than
# depend on those internals, this version (a) adds the new tag with the
# lowest-level, most stable tokenizer method (`add_tokens`), and (b) builds
# every encoder/decoder sequence *manually* in the NLLB convention
# `[lang_tag_id, tok_1, ..., tok_n, eos_id]`, instead of relying on
# `tokenizer.src_lang` / `tokenizer.tgt_lang` to inject the tag for us. This
# sidesteps the whole class of AttributeError/version-mismatch issues.

def register_ekegusii_tag(tok):
    """Add guz_Latn to the tokenizer's vocabulary as an atomic special token.
    `add_tokens` is a low-level, stable API that doesn't touch
    `additional_special_tokens` at all, unlike `add_special_tokens(...)`."""
    if EKEGUSII_TAG not in tok.get_vocab():
        tok.add_tokens([EKEGUSII_TAG], special_tokens=True)

register_ekegusii_tag(tokenizer)
EKEGUSII_TAG_ID = tokenizer.convert_tokens_to_ids(EKEGUSII_TAG)
SWAHILI_TAG_ID = tokenizer.convert_tokens_to_ids('swh_Latn')
EOS_ID = tokenizer.eos_token_id

bleu_metric = evaluate.load('sacrebleu')
chrf_metric = evaluate.load('chrf')

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)


def build_model():
    """Fresh base model + resized/warm-started Ekegusii embedding + LoRA adapter.
    Called once per architecture so each experiment starts from an identical,
    correctly-configured base -- this is what makes the 3-way comparison fair.
    """
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
    model.resize_token_embeddings(len(tokenizer))
    with torch.no_grad():
        in_emb = model.get_input_embeddings()
        in_emb.weight[EKEGUSII_TAG_ID] = in_emb.weight[SWAHILI_TAG_ID].clone()
        out_emb = model.get_output_embeddings()
        if out_emb is not None and out_emb.weight.shape[0] == in_emb.weight.shape[0]:
            out_emb.weight[EKEGUSII_TAG_ID] = out_emb.weight[SWAHILI_TAG_ID].clone()

    # --- FIX: force the decoder to condition on the target-language tag at
    # generation time. Without this, model.generate() (called internally by
    # Seq2SeqTrainer.evaluate/predict) has no idea which language to decode --
    # this alone was likely enough to sink every architecture's score in the
    # original run.
    #
    # NOTE (v3 patch): this MUST be set on `model.generation_config`, not
    # `model.config`. Newer transformers releases raise a hard ValueError if
    # generation parameters are found on `model.config` at all ("You have
    # modified the pretrained model configuration to control generation ...
    # This strategy to control generation is not supported anymore"). The two
    # objects look interchangeable in older tutorials but are not anymore --
    # `generation_config` is now the only place `.generate()` will read this from.
    model.generation_config.forced_bos_token_id = EKEGUSII_TAG_ID

    peft_model = get_peft_model(model, peft_config)
    # Trade a bit of speed for a lot of activation memory -- cheap insurance
    # against OOM on a GPU that may be shared with other tenants/kernels.
    peft_model.gradient_checkpointing_enable()
    peft_model.enable_input_require_grads()
    return peft_model


def clear_gpu():
    """Call after `del <model, trainer, ...>` at the end of each architecture
    cell -- NOT a substitute for `del`. Deleting variables *inside* this
    function wouldn't drop the caller's own reference (Python's `del` only
    removes the name binding it's given, not the caller's), so each
    architecture cell explicitly `del`s its own model/trainer/datasets first
    and then calls this to actually reclaim the freed CUDA memory.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _encode_with_tag(texts, tag_id, max_length):
    """Manually build NLLB-format sequences [tag_id, tok_1, ..., tok_n, eos_id],
    bypassing tokenizer.src_lang/tgt_lang entirely (see note above)."""
    enc = tokenizer(list(texts), add_special_tokens=False, max_length=max_length - 2, truncation=True)
    input_ids = [[tag_id] + ids + [EOS_ID] for ids in enc['input_ids']]
    attention_mask = [[1] * len(ids) for ids in input_ids]
    return input_ids, attention_mask


def preprocess_nmt_function(examples, src_lang='English', tgt_lang='Ekegusii'):
    inputs = [str(x) for x in examples[src_lang]]
    targets = [str(x) for x in examples[tgt_lang]]
    src_tag_id = tokenizer.convert_tokens_to_ids(LANG_TAGS.get(src_lang, 'eng_Latn'))
    tgt_tag_id = tokenizer.convert_tokens_to_ids(LANG_TAGS.get(tgt_lang, EKEGUSII_TAG))

    input_ids, attention_mask = _encode_with_tag(inputs, src_tag_id, max_length=128)
    labels, _ = _encode_with_tag(targets, tgt_tag_id, max_length=128)

    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels_wrapped = [[l.strip()] for l in decoded_labels]
    bleu = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels_wrapped)
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels_wrapped)
    return {'bleu': bleu['score'], 'chrf': chrf['score']}


def make_training_args(output_dir, epochs, train_bs=16, eval_bs=16, with_eval=True, lr=3e-4):
    # NOTE: train_bs/eval_bs default to 16 (not the original 32) and beam
    # search defaults to greedy (num_beams=1, not 4) as headroom against OOM --
    # this GPU may be shared with other kernels/tenants (see the memory note
    # in section 4). Bump these back up once a run completes cleanly and you
    # can see how much free VRAM you actually have via `!nvidia-smi`.
    kwargs = dict(
        output_dir=output_dir,
        eval_strategy='epoch' if with_eval else 'no',
        save_strategy='epoch',
        learning_rate=lr,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=epochs,
        # gradient checkpointing is enabled directly on the model in
        # build_model() (model.gradient_checkpointing_enable() +
        # enable_input_require_grads(), the combination PEFT actually needs);
        # not duplicated here as a TrainingArguments flag to avoid two
        # separate code paths trying to toggle the same thing.
        predict_with_generate=True,
        generation_max_length=128,
        generation_num_beams=1,
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=20,
        save_total_limit=2,
        report_to='none',
    )
    if with_eval:
        kwargs.update(load_best_model_at_end=True, metric_for_best_model='chrf', greater_is_better=True)
    return Seq2SeqTrainingArguments(**kwargs)


def evaluate_on_test(model, test_df, src_lang='English', tgt_lang='Ekegusii'):
    test_ds = Dataset.from_pandas(test_df).map(
        lambda x: preprocess_nmt_function(x, src_lang, tgt_lang), batched=True,
    )
    args = make_training_args('./tmp_eval', epochs=1, with_eval=False)
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
        compute_metrics=compute_metrics,
    )
    results = trainer.evaluate(test_ds)
    return results.get('eval_bleu', 0.0), results.get('eval_chrf', 0.0)


print('[OK] Ekegusii tag registered:', EKEGUSII_TAG, '-> id', EKEGUSII_TAG_ID, '(warm-started from swh_Latn id', SWAHILI_TAG_ID, ')')
print('[OK] Tokenizer, model builder, and NMT evaluation functions compiled.')


## 4. Architecture 1: Direct Trilingual PSA Fine-Tuning
Fine-tunes NLLB-200 + LoRA directly on the Trilingual PSA dataset.

In [ ]:
print('=== RUNNING ARCHITECTURE 1: Direct Trilingual Fine-Tuning ===')

a1_train = pd.read_csv(os.path.join(arch1_dir, 'train.csv'))
a1_val = pd.read_csv(os.path.join(arch1_dir, 'val.csv'))
a1_test = pd.read_csv(os.path.join(arch1_dir, 'test.csv'))

a1_model = build_model()
a1_model.print_trainable_parameters()

train_ds1 = Dataset.from_pandas(a1_train).map(lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'), batched=True)
val_ds1 = Dataset.from_pandas(a1_val).map(lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'), batched=True)

args1 = make_training_args('./output_arch1', epochs=3)

trainer1 = Seq2SeqTrainer(
    model=a1_model,
    args=args1,
    train_dataset=train_ds1,
    eval_dataset=val_ds1,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a1_model),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer1.train()

bleu1, chrf1 = evaluate_on_test(a1_model, a1_test)

print(f"\n🏆 Architecture 1 Final Test Results:")
print(f'SacreBLEU = {bleu1:.2f}')
print(f'chrF       = {chrf1:.2f}')

# Free Architecture 1's model/trainer before Architecture 2 loads a second
# full copy of NLLB-200 onto the same GPU (see the OOM note in section 4).
del a1_model, trainer1, train_ds1, val_ds1
clear_gpu()


## 5. Architecture 2: Sequential Transfer Learning
Fine-tunes on Bilingual Data → Trilingual Bible Data → Trilingual PSA Data. Unlike the original run, every
stage now evaluates against the shared PSA validation set, so `arch2_stage_history` gives a real per-stage
learning curve instead of a single end-of-pipeline number.

In [ ]:
print('=== RUNNING ARCHITECTURE 2: Sequential Transfer Learning ===')
bi_file = find_data_file('English_Ekegusii_Web_News.csv')
bible_file = find_data_file('bibile.csv')
a2_bi = pd.read_csv(bi_file)
a2_bible = pd.read_csv(bible_file)
a2_train, a2_val, a2_test = a1_train, a1_val, a1_test

a2_model = build_model()
arch2_stage_history = []

# Stage 1: Bilingual
print('-> Stage 1: Fine-tuning on Bilingual Data...')
ds2_1 = Dataset.from_pandas(a2_bi).map(lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'), batched=True)
args2_1 = make_training_args('./output_arch2_s1', epochs=1, with_eval=False)
Seq2SeqTrainer(model=a2_model, args=args2_1, train_dataset=ds2_1, processing_class=tokenizer,
               data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a2_model)).train()
b, c = evaluate_on_test(a2_model, a2_val)
arch2_stage_history.append({'stage': '1. Bilingual', 'BLEU': b, 'chrF': c})
print(f'   stage 1 val -> BLEU {b:.2f} | chrF {c:.2f}')

# Stage 2: Bible
print('-> Stage 2: Fine-tuning on Trilingual Bible Data...')
ds2_2 = Dataset.from_pandas(a2_bible.sample(min(4000, len(a2_bible)), random_state=42)).map(
    lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'), batched=True)
args2_2 = make_training_args('./output_arch2_s2', epochs=1, with_eval=False)
Seq2SeqTrainer(model=a2_model, args=args2_2, train_dataset=ds2_2, processing_class=tokenizer,
               data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a2_model)).train()
b, c = evaluate_on_test(a2_model, a2_val)
arch2_stage_history.append({'stage': '2. Bible', 'BLEU': b, 'chrF': c})
print(f'   stage 2 val -> BLEU {b:.2f} | chrF {c:.2f}')

# Stage 3: PSA Adaptation
print('-> Stage 3: Domain Adaptation on Trilingual PSA Data...')
ds2_3 = Dataset.from_pandas(a2_train).map(lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'), batched=True)
ds2_val = Dataset.from_pandas(a2_val).map(lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'), batched=True)
args2_3 = make_training_args('./output_arch2_s3', epochs=3)
trainer2 = Seq2SeqTrainer(model=a2_model, args=args2_3, train_dataset=ds2_3, eval_dataset=ds2_val,
                           processing_class=tokenizer,
                           data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a2_model),
                           compute_metrics=compute_metrics,
                           callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer2.train()
b, c = evaluate_on_test(a2_model, a2_val)
arch2_stage_history.append({'stage': '3. PSA', 'BLEU': b, 'chrF': c})

bleu2, chrf2 = evaluate_on_test(a2_model, a2_test)
print(f'\n🏆 Architecture 2 Final Test Results: SacreBLEU = {bleu2:.2f} | chrF = {chrf2:.2f}')
display(pd.DataFrame(arch2_stage_history))

# Free Architecture 2's model/trainer -- ds2_1/ds2_2/ds2_3/ds2_val stay alive,
# Architecture 3 reuses them for a fair like-for-like comparison.
del a2_model, trainer2
clear_gpu()


## 6. Architecture 3: Progressive Curriculum Transfer Learning
Adapts through 4 stages: Unilingual PSA denoising pre-adaptation → Bilingual → Trilingual Bible → Trilingual
PSA. Stage 1 is rebuilt from scratch (see changelog item 4): the original notebook faked a translation pair
by copying English into the Ekegusii column; this version instead trains a real denoising objective
entirely in English (`eng_Latn -> eng_Latn`) to adapt the encoder to PSA-register vocabulary before any
cross-lingual stage begins.

In [ ]:
print('=== RUNNING ARCHITECTURE 3: Progressive Curriculum Transfer Learning ===')
uni_file = find_data_file('english_unilingual.csv')
a3_uni = pd.read_csv(uni_file)

a3_model = build_model()
arch3_stage_history = []


def corrupt_sentence(text, drop_prob=0.15, shuffle_window=3, seed=None):
    """Light BART-style noising: random word dropout + local shuffling.
    Used to build a denoising self-supervision signal from monolingual text
    -- NOT a stand-in translation pair like the original notebook's
    verbatim-copy trick, which just taught the model to echo its input.
    """
    rng = random.Random(seed)
    words = str(text).split()
    kept = [w for w in words if rng.random() > drop_prob]
    if not kept:
        kept = words[:1] if words else ['']
    for i in range(0, len(kept), shuffle_window):
        window = kept[i:i + shuffle_window]
        rng.shuffle(window)
        kept[i:i + shuffle_window] = window
    return ' '.join(kept)


# Stage 1: Unilingual denoising domain-adaptive pretraining (English -> English)
print('-> Stage 1: Unilingual PSA denoising pre-adaptation (English -> English)...')
uni_sample = a3_uni.sample(min(4000, len(a3_uni)), random_state=42).reset_index(drop=True)
uni_sample['English_clean'] = uni_sample['English']
uni_sample['English_noisy'] = [corrupt_sentence(t, seed=i) for i, t in enumerate(uni_sample['English'])]

# Temporarily register 'noisy'/'clean' as English<->English so this stage
# never touches the Ekegusii tag -- it's pure monolingual domain adaptation.
_orig_lang_tags = dict(LANG_TAGS)
LANG_TAGS['noisy'] = 'eng_Latn'
LANG_TAGS['clean'] = 'eng_Latn'
ds3_1 = Dataset.from_pandas(uni_sample).map(
    lambda x: preprocess_nmt_function({'noisy': x['English_noisy'], 'clean': x['English_clean']},
                                       src_lang='noisy', tgt_lang='clean'),
    batched=True,
)
LANG_TAGS.clear(); LANG_TAGS.update(_orig_lang_tags)  # restore English/Kiswahili/Ekegusii mapping

args3_1 = make_training_args('./output_arch3_s1', epochs=1, with_eval=False)
Seq2SeqTrainer(model=a3_model, args=args3_1, train_dataset=ds3_1, processing_class=tokenizer,
               data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a3_model)).train()
b, c = evaluate_on_test(a3_model, a2_val)
arch3_stage_history.append({'stage': '1. Unilingual denoise', 'BLEU': b, 'chrF': c})
print(f'   stage 1 val (English->Ekegusii, before any cross-lingual training) -> BLEU {b:.2f} | chrF {c:.2f}')

# Stage 2: Bilingual (reuse Architecture 2's stage-1 dataset/args for a fair comparison)
print('-> Stage 2: Bilingual Data Fine-tuning...')
Seq2SeqTrainer(model=a3_model, args=args2_1, train_dataset=ds2_1, processing_class=tokenizer,
               data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a3_model)).train()
b, c = evaluate_on_test(a3_model, a2_val)
arch3_stage_history.append({'stage': '2. Bilingual', 'BLEU': b, 'chrF': c})
print(f'   stage 2 val -> BLEU {b:.2f} | chrF {c:.2f}')

# Stage 3: Bible
print('-> Stage 3: Bible Data Transfer...')
Seq2SeqTrainer(model=a3_model, args=args2_2, train_dataset=ds2_2, processing_class=tokenizer,
               data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a3_model)).train()
b, c = evaluate_on_test(a3_model, a2_val)
arch3_stage_history.append({'stage': '3. Bible', 'BLEU': b, 'chrF': c})
print(f'   stage 3 val -> BLEU {b:.2f} | chrF {c:.2f}')

# Stage 4: PSA Adaptation
print('-> Stage 4: Final Trilingual PSA Adaptation...')
args3_4 = make_training_args('./output_arch3_s4', epochs=3)
trainer3 = Seq2SeqTrainer(model=a3_model, args=args3_4, train_dataset=ds2_3, eval_dataset=ds2_val,
                           processing_class=tokenizer,
                           data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a3_model),
                           compute_metrics=compute_metrics,
                           callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer3.train()
b, c = evaluate_on_test(a3_model, a2_val)
arch3_stage_history.append({'stage': '4. PSA', 'BLEU': b, 'chrF': c})

bleu3, chrf3 = evaluate_on_test(a3_model, a1_test)
print(f'\n🏆 Architecture 3 Final Test Results: SacreBLEU = {bleu3:.2f} | chrF = {chrf3:.2f}')
display(pd.DataFrame(arch3_stage_history))

# Nothing after this needs the models or per-stage datasets, only the scalar
# bleu/chrf results and the stage_history lists already captured above.
del a3_model, trainer3, ds3_1, ds2_1, ds2_2, ds2_3, ds2_val
clear_gpu()


## 7. Comparative Evaluation & Benchmark Visualizations
Final SacreBLEU/chrF comparison across all 3 architectures, plus the per-stage learning curves for
Architectures 2 and 3 that the original notebook couldn't produce (no intermediate validation).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

results_data = [
    {'Architecture': 'Arch 1 (Direct PSA)', 'SacreBLEU': round(bleu1, 2), 'chrF': round(chrf1, 2)},
    {'Architecture': 'Arch 2 (Bilingual -> Bible -> PSA)', 'SacreBLEU': round(bleu2, 2), 'chrF': round(chrf2, 2)},
    {'Architecture': 'Arch 3 (Curriculum Transfer)', 'SacreBLEU': round(bleu3, 2), 'chrF': round(chrf3, 2)},
]

df_results = pd.DataFrame(results_data)
print('=== 📊 FINAL ARCHITECTURE BENCHMARK EVALUATION TABLE ===')
display(df_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=df_results, x='Architecture', y='SacreBLEU', ax=axes[0], palette='viridis')
axes[0].set_title('SacreBLEU Score Comparison (Higher is Better)', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)
sns.barplot(data=df_results, x='Architecture', y='chrF', ax=axes[1], palette='magma')
axes[1].set_title('chrF Score Comparison (Higher is Better)', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

# Per-stage learning curves for the curriculum architectures
df_a2 = pd.DataFrame(arch2_stage_history)
df_a3 = pd.DataFrame(arch3_stage_history)
print('\n=== Architecture 2 per-stage validation history ===')
display(df_a2)
print('=== Architecture 3 per-stage validation history ===')
display(df_a3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(df_a2['stage'], df_a2['chrF'], marker='o', label='Arch 2')
axes[0].plot(df_a3['stage'].str.replace(r'^\d+\. ', '', regex=True),
             df_a3['chrF'], marker='s', label='Arch 3')
axes[0].set_title('chrF by curriculum stage (val set)')
axes[0].set_ylabel('chrF')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend()

axes[1].plot(df_a2['stage'], df_a2['BLEU'], marker='o', label='Arch 2')
axes[1].plot(df_a3['stage'].str.replace(r'^\d+\. ', '', regex=True),
             df_a3['BLEU'], marker='s', label='Arch 3')
axes[1].set_title('SacreBLEU by curriculum stage (val set)')
axes[1].set_ylabel('SacreBLEU')
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend()

plt.tight_layout()
plt.show()
